In [2]:
import pandas as pd 
df = pd.read_csv('Data/drug_review_train_clean.csv')

df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Data/drug_review_train_clean.csv'

In [2]:
df.drop(columns=['Unnamed: 0','date', 'usefulCount', 'review_length'], inplace=True)

In [1]:
import pandas as pd
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import nltk
import re
import numpy as np



# Load data
df = pd.read_csv("/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv")  # replace with your actual path
df.drop(columns=['Unnamed: 0','date', 'usefulCount', 'review_length'], inplace=True)
# Drop rows with missing reviews or ratings
df = df.dropna(subset=['review', 'rating'])
df.drop(['patient_id', 'condition'], axis=1, inplace=True)

In [8]:
# === 1. Load data ===
train_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv')
test_df = pd.read_csv('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_test_clean.csv')

# Drop unneeded columns
train_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)
test_df.drop(['Unnamed: 0', 'date', 'usefulCount', 'review_length'], axis=1, inplace=True)

# === 2. Feature Extraction (BoW ) ===
vectorizer = TfidfVectorizer()
X_train_bow = vectorizer.fit_transform(train_df['review_clean'])
X_test_bow = vectorizer.transform(test_df['review_clean'])

y_train = train_df['rating'].values
y_test = test_df['rating'].values
vocab = vectorizer.get_feature_names_out()

from sklearn.decomposition import PCA

# Step 7: Reduce dimensionality with PCA
n_components = 50  # You can adjust this
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_train_bow)

# OPTIONAL: View explained variance
print("Explained variance ratio:", pca.explained_variance_ratio_)

# Step 8: Train/test split on PCA-transformed data
X_train, X_test, y_train, y_test = train_test_split(X_pca, df['rating'], test_size=0.2, random_state=42)

# Step 9: Train the Lasso model
lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

# Step 10: Predict and evaluate
y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

Explained variance ratio: [0.01132349 0.00607778 0.00557468 0.00482799 0.00464283 0.0041493
 0.00406108 0.00386667 0.00376416 0.00336787 0.00312784 0.00306189
 0.0030377  0.0028748  0.00281791 0.00279925 0.00270906 0.00266094
 0.0026083  0.0025662  0.00250849 0.00248655 0.00237068 0.00233278
 0.00225567 0.00224993 0.00223476 0.00221697 0.00219491 0.00216227
 0.00213467 0.00211626 0.00204255 0.00200131 0.00197787 0.00194068
 0.00189731 0.00189165 0.00187568 0.00182962 0.00182314 0.00181744
 0.00180133 0.00179004 0.00177249 0.00173053 0.00171887 0.00170802
 0.00168298 0.00166713]
MSE: 10.485177682565723


In [9]:
from sklearn.decomposition import PCA

# Step 7: Reduce dimensionality with PCA
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_train_bow)

# OPTIONAL: View explained variance
print("Explained variance ratio:", pca.explained_variance_ratio_)

# Step 8: Train/test split on PCA-transformed data
X_train, X_test, y_train, y_test = train_test_split(X_pca, df['rating'], test_size=0.2, random_state=42)

# Step 9: Train the Lasso model
lasso = Lasso(alpha=0.1, max_iter=1000)
lasso.fit(X_train, y_train)

# Step 10: Predict and evaluate
y_pred = lasso.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))

Explained variance ratio: [0.01132349 0.00607778 0.00557468 0.00482799 0.00464283 0.0041493
 0.00406108 0.00386667 0.00376416 0.00336787 0.00312784 0.00306189
 0.0030377  0.0028748  0.00281791 0.00279925 0.00270906 0.00266094
 0.0026083  0.0025662  0.00250849 0.00248655 0.00237068 0.00233278
 0.00225567 0.00224993 0.00223476 0.00221697 0.00219491 0.00216227
 0.00213467 0.00211626 0.00204255 0.00200131 0.00197787 0.00194068
 0.00189731 0.00189165 0.00187568 0.00182962 0.00182314 0.00181744
 0.00180133 0.00179004 0.00177249 0.00173053 0.00171887 0.00170802
 0.00168298 0.00166713]
MSE: 10.485177682565723


In [7]:
import numpy as np

n_top_words = 10  # number of words to show per component

for i, component in enumerate(pca.components_):
    print(f"\nPCA Component {i + 1}:")
    
    # Get indices of top positive values
    top_pos_indices = component.argsort()[-n_top_words:][::-1]
    # Get indices of top negative values
    top_neg_indices = component.argsort()[:n_top_words]
    
    print("  Top positive words:")
    for idx in top_pos_indices:
        print(f"    {vocab[idx]} ({component[idx]:.4f})")
    
    print("  Top negative words:")
    for idx in top_neg_indices:
        print(f"    {vocab[idx]} ({component[idx]:.4f})")



PCA Component 1:
  Top positive words:
    period (0.4608)
    pill (0.2774)
    month (0.2520)
    birth (0.2428)
    control (0.2196)
    gain (0.1716)
    get (0.1610)
    weight (0.1586)
    acne (0.1514)
    cramp (0.1443)
  Top negative words:
    mg (-0.1766)
    sleep (-0.1117)
    pain (-0.1025)
    medication (-0.0903)
    medicine (-0.0873)
    anxiety (-0.0824)
    night (-0.0738)
    work (-0.0722)
    dose (-0.0693)
    drug (-0.0648)

PCA Component 2:
  Top positive words:
    pain (0.4317)
    skin (0.0647)
    cramp (0.0632)
    use (0.0610)
    relief (0.0568)
    infection (0.0535)
    painful (0.0508)
    product (0.0503)
    insertion (0.0499)
    knee (0.0490)
  Top negative words:
    anxiety (-0.3200)
    mg (-0.2891)
    feel (-0.2565)
    weight (-0.2002)
    depression (-0.1822)
    start (-0.1662)
    lose (-0.1488)
    gain (-0.1473)
    sleep (-0.1365)
    week (-0.1295)

PCA Component 3:
  Top positive words:
    day (0.4214)
    pain (0.2750)
    period

In [1]:
from linearmodel import train_lasso_model
PATH_TRAIN = '/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv'
PATH_TEST = '/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_test_clean.csv'
model, vocab, X_test, y_test = train_lasso_model('/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_train_clean.csv','/Users/youssefbenmansour/Desktop/HCML-NLP-Project/Data/drug_review_test_clean.csv')

In [2]:
from linearmodel import print_lasso_results

print_lasso_results(model, vocab, X_test, y_test)

Test MSE with LASSO: 8.0382
Number of non-zero features: 91

Top 20 LASSO features:
PCA Component 100 (from BoW): 0.9982
PCA Component 30 (from BoW): -0.8278
PCA Component 27 (from BoW): 0.8059
PCA Component 69 (from BoW): 0.6993
PCA Component 76 (from BoW): -0.6358
PCA Component 85 (from BoW): 0.6355
PCA Component 33 (from BoW): -0.5163
PCA Component 96 (from BoW): 0.5108
PCA Component 23 (from BoW): -0.5098
PCA Component 43 (from BoW): -0.5082
PCA Component 88 (from BoW): -0.4411
PCA Component 5 (from BoW): -0.4307
PCA Component 73 (from BoW): -0.4172
PCA Component 57 (from BoW): 0.3656
PCA Component 34 (from BoW): 0.3628
PCA Component 70 (from BoW): -0.3386
PCA Component 24 (from BoW): 0.3359
PCA Component 92 (from BoW): -0.3232
PCA Component 79 (from BoW): 0.3205
PCA Component 94 (from BoW): -0.3026


In [22]:
from sklearn.metrics import mean_squared_error, accuracy_score

def print_accuracy(model, X_test, y_test):
    y_pred = model.predict(X_test)
     # Compute and print accuracy (after rounding predictions to nearest integer)
    y_pred_rounded = np.clip(np.round(y_pred), 1, 10).astype(int)  # ratings are likely between 1 and 10
    y_test_int = y_test.astype(int)
    acc = accuracy_score(y_test_int, y_pred_rounded)
    print(f"Test Accuracy (rounded predictions): {acc * 100:.2f}%")


print_accuracy(model, X_test, y_test)

Test Accuracy (rounded predictions): 12.45%
